<a href="https://colab.research.google.com/github/ignaciocolussi/HAcore/blob/dev/notebooks/unit2/llamaindex/agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Agents in LlamaIndex

This notebook is part of the [Hugging Face Agents Course](https://www.hf.co/learn/agents-course), a free Course from beginner to expert, where you learn to build Agents.

![Agents course share](https://huggingface.co/datasets/agents-course/course-images/resolve/main/en/communication/share.png)

## Let's install the dependencies

We will install the dependencies for this unit.

In [18]:
!pip install llama-index datasets llama-index-callbacks-arize-phoenix llama-index-vector-stores-chroma llama-index-llms-huggingface-api llama-index-llms-together -U -q

And, let's log in to Hugging Face to use serverless Inference APIs.

In [2]:
from huggingface_hub import login

login()

## Initialising agents

Let's start by initialising an agent. We will use the basic `AgentWorkflow` class to create an agent.

In [20]:
from llama_index.llms.together import TogetherLLM
from llama_index.core.agent.workflow import AgentWorkflow, ToolCallResult, AgentStream


def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers"""
    return a - b


def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b


def divide(a: int, b: int) -> int:
    """Divide two numbers"""
    return a / b


llm = TogetherLLM(model="Qwen/Qwen2.5-Coder-32B-Instruct", api_key="13bdcbbc507e3f2f89cb354886791129bbf8e38c497443b8fd80c051d2949dc9")

agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[subtract, multiply, divide, add],
    llm=llm,
    system_prompt="You are a math agent that can add, subtract, multiply, and divide numbers using provided tools.",
)

Then, we can run the agent and get the response and reasoning behind the tool calls.

In [21]:
handler = agent.run("What is (2 + 2) * 2?")
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: add
Action Input: {"a": 2, "b": 2}
Called tool:  add {'a': 2, 'b': 2} => 4
Thought: Now I need to multiply the result by 2.
Action: multiply
Action Input: {'a': 4, 'b': 2}
Called tool:  multiply {'a': 4, 'b': 2} => 8
Thought: I can answer without using any more tools. I'll use the user's language to answer
Answer: (2 + 2) * 2 = 8

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='(2 + 2) * 2 = 8')]), tool_calls=[ToolCallResult(tool_name='add', tool_kwargs={'a': 2, 'b': 2}, tool_id='08dae7a7-26e3-4e6e-be68-23397e6e7cf2', tool_output=ToolOutput(content='4', tool_name='add', raw_input={'args': (), 'kwargs': {'a': 2, 'b': 2}}, raw_output=4, is_error=False), return_direct=False), ToolCallResult(tool_name='multiply', tool_kwargs={'a': 4, 'b': 2}, tool_id='f06792ac-c768-49b6-a7c0-e5c18f21c33a', tool_output=ToolOutput(content='8', tool_name='multiply', raw_input={'args': (), 'kwargs': {'a': 4, 'b': 2}}, raw_output=8, is_error=False), return_direct=False)], raw={'id': 'nm82KxP-4yUbBN-920cd56b2e517bcd', 'choices': [{'delta': {'content': '', 'function_call': None, 'refusal': None, 'role': 'assistant', 'tool_calls': None, 'token_id': 151645}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'text': '', 'seed': 645154097812031700

In a similar fashion, we can pass state and context to the agent.


In [22]:
from llama_index.core.workflow import Context

ctx = Context(agent)

response = await agent.run("My name is Bob.", ctx=ctx)
response = await agent.run("What was my name again?", ctx=ctx)
response

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Your name is Bob.')]), tool_calls=[], raw={'id': 'nm82eRq-4yUbBN-920cd6eff9d87c2d', 'choices': [{'delta': {'content': '', 'function_call': None, 'refusal': None, 'role': 'assistant', 'tool_calls': None, 'token_id': 151645}, 'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'text': '', 'seed': 14730045666860230000}], 'created': 1742050398, 'model': 'Qwen/Qwen2.5-Coder-32B-Instruct', 'object': 'chat.completion.chunk', 'service_tier': None, 'system_fingerprint': None, 'usage': {'completion_tokens': 29, 'prompt_tokens': 869, 'total_tokens': 898, 'completion_tokens_details': None, 'prompt_tokens_details': None}}, current_agent_name='Agent')

## Creating RAG Agents with QueryEngineTools

Let's now re-use the `QueryEngine` we defined in the [previous unit on tools](/tools.ipynb) and convert it into a `QueryEngineTool`. We will pass it to the `AgentWorkflow` class to create a RAG agent.

In [25]:
!pip install llama-index-embeddings-together

In [31]:
import chromadb

from llama_index.core import VectorStoreIndex
from llama_index.llms.together import TogetherLLM
from llama_index.embeddings.together import TogetherEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.vector_stores.chroma import ChromaVectorStore

# Create a vector store
db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("alfred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

# Create a query engine
embed_model = TogetherEmbedding(model_name="BAAI/bge-base-en-v1.5", api_key=api_key)
llm = TogetherLLM(model="Qwen/Qwen2.5-Coder-32B-Instruct", api_key=api_key)
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store, embed_model=embed_model
)
query_engine = index.as_query_engine(llm=llm)
query_engine_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="personas",
    description="descriptions for various types of personas",
    return_direct=False,
)

# Create a RAG agent
query_engine_agent = AgentWorkflow.from_tools_or_functions(
    tools_or_functions=[query_engine_tool],
    llm=llm,
    system_prompt="You are a helpful assistant that has access to a database containing persona descriptions. ",
)

And, we can once more get the response and reasoning behind the tool calls.

In [32]:
handler = query_engine_agent.run(
    "Search the database for 'science fiction' and return some persona descriptions."
)
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: personas
Action Input: {"input": "science fiction"}
Called tool:  personas {'input': 'science fiction'} => Empty Response
Thought: It seems there are no specific persona descriptions for 'science fiction' in the database. I'll try to provide a general approach to creating personas related to science fiction based on common themes and characters found in the genre.
Action: personas
Action Input: {'input': 'general science fiction themes'}
Called tool:  personas {'input': 'general science fiction themes'} => Empty Response
Thought: It appears that the database does not contain specific or general persona descriptions related to science fiction themes. Since I cannot retrieve the required information from the database, I will create some example personas based on common archetypes found in science fiction literature and media.

Thought: I can answer without using any more to

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='Here are some example science fiction personas based on common archetypes:\n\n1. **The Futuristic Scientist**: A brilliant and eccentric scientist who is always pushing the boundaries of what is possible. They are often portrayed as having a deep understanding of advanced technologies and are driven by a desire to improve humanity.\n\n2. **The Rebel Pilot**: A skilled and daring pilot who fights against oppressive regimes or corporations. They are often portrayed as having a strong sense of justice and a willingness to take risks to achieve their goals.\n\n3. **The Alien Diplomat**: A being from another planet who is tasked with maintaining peace and understanding between different species. They are often portrayed as having a calm and diplomatic demeanor, but they can also be fiercely protective of their people.\n\n4. **The Cybernetic Warrior**: 

## Creating multi-agent systems

We can also create multi-agent systems by passing multiple agents to the `AgentWorkflow` class.

In [35]:
from llama_index.core.agent.workflow import (
    AgentWorkflow,
    ReActAgent,
)


# Define some tools
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


def subtract(a: int, b: int) -> int:
    """Subtract two numbers."""
    return a - b


# Create agent configs
# NOTE: we can use FunctionAgent or ReActAgent here.
# FunctionAgent works for LLMs with a function calling API.
# ReActAgent works for any LLM.
calculator_agent = ReActAgent(
    name="calculator",
    description="Performs basic arithmetic operations",
    system_prompt="You are a calculator assistant. Use your tools for any math operation.",
    tools=[add, subtract],
    llm=llm,
)

query_agent = ReActAgent(
    name="info_lookup",
    description="Looks up information about XYZ",
    system_prompt="Use your tool to query a RAG system to answer information about XYZ",
    tools=[query_engine_tool],
    llm=llm,
)

# Create and run the workflow
agent = AgentWorkflow(agents=[calculator_agent, query_agent], root_agent="calculator")

# Run the system
handler = agent.run(user_msg="Can you tell me what hapend to romania in year 1956?")

In [36]:
async for ev in handler.stream_events():
    if isinstance(ev, ToolCallResult):
        print("")
        print("Called tool: ", ev.tool_name, ev.tool_kwargs, "=>", ev.tool_output)
    elif isinstance(ev, AgentStream):  # showing the thought process
        print(ev.delta, end="", flush=True)

resp = await handler
resp

Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: handoff
Action Input: {"to_agent": "info_lookup", "reason": "The user is asking for historical information about Romania in 1956, which is not a math operation."}
Called tool:  handoff {'to_agent': 'info_lookup', 'reason': 'The user is asking for historical information about Romania in 1956, which is not a math operation.'} => Agent info_lookup is now handling the request due to the following reason: The user is asking for historical information about Romania in 1956, which is not a math operation..
Please continue with the current request.
Thought: The current language of the user is: English. I need to use a tool to help me answer the question.
Action: personas
Action Input: {"input": "historical events in Romania in 1956"}
Called tool:  personas {'input': 'historical events in Romania in 1956'} => Empty Response
Thought: I cannot answer the question with the provided t

AgentOutput(response=ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='There seems to be no specific major historical event that stands out for Romania in 1956. However, it was a year during the communist rule under Nicolae Ceaușescu, and the country continued to follow the policies set by the Soviet Union. If you have more specific details or another year in mind, feel free to ask!')]), tool_calls=[ToolCallResult(tool_name='handoff', tool_kwargs={'to_agent': 'info_lookup', 'reason': 'The user is asking for historical information about Romania in 1956, which is not a math operation.'}, tool_id='8c38d646-3fea-4107-ba91-d6a4a545ec73', tool_output=ToolOutput(content='Agent info_lookup is now handling the request due to the following reason: The user is asking for historical information about Romania in 1956, which is not a math operation..\nPlease continue with the current request.', tool_name='handoff', raw_input={'arg